In [3]:
import gradio as gr
from pypdf import PdfReader
import ollama
import os
import json
import matplotlib.pyplot as plt
import datetime
import random

# -------------------------
# Storage & Persistence
# -------------------------
DATA_FILE = "study_planner_data.json"
PLAN_FILE = "study_plan.txt"

def load_data():
    if os.path.exists(DATA_FILE):
        try:
            with open(DATA_FILE, "r") as f:
                return json.load(f)
        except Exception:
            pass
    return {"plan": {}, "goals": []}

def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=4)

# -------------------------
# Read PDF
# -------------------------
def read_pdf(file):
    if file is None:
        return ""

    reader = PdfReader(file.name)
    text = ""

    for page in reader.pages:
        try:
            text += page.extract_text() + "\n"
        except Exception:
            pass

    return text

# -------------------------
# AI Answer
# -------------------------
def ask_ai(context, question):
    prompt = f"""
You are a helpful Student Study Assistant.

Study Material:
{context}

Question:
{question}

Answer in simple language.
"""
    try:
        response = ollama.chat(
            model="llama3.2",
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )
        return response["message"]["content"]
    except Exception as e:
        return f"⚠️ Error communicating with Ollama: {str(e)}\n\nMake sure Ollama is installed and running locally, and the 'llama3.2' model is downloaded (run 'ollama run llama3.2' in terminal)."

# -------------------------
# Study Planner (Daily)
# -------------------------
def save_plan(name, subject, hours):
    plan = f"""
Student : {name}
Subject : {subject}
Study Hours : {hours}

Today's Plan
-------------
1. Read Chapter
2. Make Notes
3. Practice Questions
4. Revise
"""
    # Save legacy text file
    with open(PLAN_FILE, "w") as f:
        f.write(plan)
        
    # Also persist daily plan to JSON data
    data = load_data()
    data["plan"] = {
        "student": name,
        "subject": subject,
        "hours": hours,
        "plan_text": plan
    }
    save_data(data)

    return plan

def view_plan():
    if os.path.exists(PLAN_FILE):
        with open(PLAN_FILE, "r") as f:
            return f.read()
    return "No Study Plan Found."

# -------------------------
# Process AI Assistant Question
# -------------------------
def process(name, pdf_file, question):
    if pdf_file is None:
        return "Upload a PDF."

    text = read_pdf(pdf_file)

    if text.strip() == "":
        return "No readable text found."

    answer = ask_ai(text, question)
    return answer

# -------------------------
# Weekly Goals & Progress Logic
# -------------------------
def get_goals_df():
    data = load_data()
    goals = data.get("goals", [])
    rows = []
    for g in goals:
        rows.append([
            g.get("id"),
            g.get("description"),
            g.get("day"),
            g.get("estimated_hours"),
            g.get("actual_hours"),
            g.get("completed")
        ])
    return rows

def add_goal(description, day, est_hours):
    if not description.strip():
        return get_goals_df(), "Goal description cannot be empty.", generate_progress_chart()
    
    data = load_data()
    goals = data.get("goals", [])
    
    new_id = 1 if not goals else max(int(g.get("id", 0)) for g in goals) + 1
    
    new_goal = {
        "id": new_id,
        "description": description.strip(),
        "day": day,
        "estimated_hours": float(est_hours),
        "actual_hours": 0.0,
        "completed": False
    }
    
    goals.append(new_goal)
    data["goals"] = goals
    save_data(data)
    
    return get_goals_df(), f"Goal '{description}' added successfully!", generate_progress_chart()

def save_goals_from_df(df_data):
    if df_data is None:
        return "No data to save.", generate_progress_chart()
        
    goals = []
    
    # Check if df_data is a pandas DataFrame
    try:
        import pandas as pd
        is_df = isinstance(df_data, pd.DataFrame)
    except ImportError:
        is_df = False
        
    if is_df:
        raw_rows = df_data.values.tolist()
    elif isinstance(df_data, dict) and "data" in df_data:
        raw_rows = df_data["data"]
    elif isinstance(df_data, list):
        raw_rows = df_data
    else:
        return "Invalid data format.", generate_progress_chart()
        
    for row in raw_rows:
        if len(row) < 6:
            continue
        try:
            goal_id = int(row[0]) if row[0] is not None else None
            desc = str(row[1])
            day = str(row[2])
            est = float(row[3]) if (row[3] is not None and str(row[3]).strip() != "") else 0.0
            act = float(row[4]) if (row[4] is not None and str(row[4]).strip() != "") else 0.0
            
            comp_val = row[5]
            if isinstance(comp_val, str):
                comp = comp_val.lower() == "true"
            else:
                comp = bool(comp_val)
            
            goals.append({
                "id": goal_id,
                "description": desc,
                "day": day,
                "estimated_hours": est,
                "actual_hours": act,
                "completed": comp
            })
        except Exception as e:
            print(f"Skipping row {row} due to error: {e}")
            continue
            
    data = load_data()
    data["goals"] = goals
    save_data(data)
    
    return "Weekly goals saved and progress updated successfully!", generate_progress_chart()

def clear_all_goals():
    data = load_data()
    data["goals"] = []
    save_data(data)
    return get_goals_df(), "All goals cleared for the week.", generate_progress_chart()

def get_progress_summary():
    data = load_data()
    goals = data.get("goals", [])
    if not goals:
        return "### 📝 No goals set for this week yet. Use the form below to add goals!"
        
    total_goals = len(goals)
    completed_goals = sum(1 for g in goals if g.get("completed", False))
    pending_goals = total_goals - completed_goals
    
    total_est = sum(float(g.get("estimated_hours", 0.0)) for g in goals)
    total_act = sum(float(g.get("actual_hours", 0.0)) for g in goals)
    
    completion_rate = (completed_goals / total_goals) * 100 if total_goals > 0 else 0
    
    summary = f"""
### 📊 Weekly Summary & Insights
- **Total Goals Set**: **{total_goals}**
- **Completed Goals**: **{completed_goals}** ({completion_rate:.1f}%)
- **Pending Goals**: **{pending_goals}**
- **Total Estimated Hours**: **{total_est:.1f} hours**
- **Total Actual Hours Spent**: **{total_act:.1f} hours**
"""
    return summary

def generate_progress_chart():
    data = load_data()
    goals = data.get("goals", [])
    
    days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
    est_hours = {d: 0.0 for d in days}
    act_hours = {d: 0.0 for d in days}
    
    for g in goals:
        day = g.get("day", "Monday")
        if day in est_hours:
            try:
                est_hours[day] += float(g.get("estimated_hours", 0.0))
                act_hours[day] += float(g.get("actual_hours", 0.0))
            except (ValueError, TypeError):
                pass
            
    plt.style.use('ggplot')
    fig, ax = plt.subplots(figsize=(10, 4.5), dpi=150)
    
    color_est = "#818cf8"  # Slate Indigo
    color_act = "#2dd4bf"  # Bright Teal
    
    x = range(len(days))
    width = 0.35
    
    rects1 = ax.bar([i - width/2 for i in x], [est_hours[d] for d in days], width, label='Estimated Hours', color=color_est, alpha=0.9)
    rects2 = ax.bar([i + width/2 for i in x], [act_hours[d] for d in days], width, label='Actual Hours Spent', color=color_act, alpha=0.9)
    
    ax.set_ylabel('Hours', fontsize=12, fontweight='bold', color='#f8fafc')
    ax.set_title('Weekly Study Hours: Estimated vs. Actual', fontsize=14, fontweight='bold', pad=20, color='#f8fafc')
    ax.set_xticks(x)
    ax.set_xticklabels(days, rotation=15, fontsize=10, color='#cbd5e1')
    
    legend = ax.legend(frameon=True, facecolor='#1e293b', edgecolor='#334155')
    for text in legend.get_texts():
        text.set_color('#f8fafc')
        
    fig.patch.set_facecolor('#0f172a')
    ax.set_facecolor('#1e293b')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('#334155')
    ax.spines['bottom'].set_color('#334155')
    ax.tick_params(colors='#cbd5e1')
    ax.yaxis.grid(True, linestyle='--', alpha=0.2, color='#475569')
    ax.xaxis.grid(False)
    
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            if height > 0:
                ax.annotate(f'{height:.1f}h',
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=9, fontweight='semibold', color='#f8fafc')
                            
    autolabel(rects1)
    autolabel(rects2)
    
    plt.tight_layout()
    chart_path = "weekly_progress_chart.png"
    plt.savefig(chart_path, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    return chart_path

def init_data():
    df_rows = get_goals_df()
    summary = get_progress_summary()
    chart = generate_progress_chart()
    return df_rows, summary, chart

# -------------------------
# Mock Test Logic & Storage
# -------------------------
def generate_fallback_questions(topic, num_questions):
    topics = {
        "math": [
            {"question": "What is the derivative of x^2?", "options": ["A) x", "B) 2x", "C) 2", "D) x^3"], "correct_answer": "B"},
            {"question": "Solve for x: 2x + 5 = 15.", "options": ["A) 5", "B) 10", "C) 15", "D) 20"], "correct_answer": "A"},
            {"question": "What is the area of a circle with radius r?", "options": ["A) 2*pi*r", "B) pi*r^2", "C) pi*d", "D) 2*pi*r^2"], "correct_answer": "B"},
            {"question": "What is the value of log(100) to base 10?", "options": ["A) 1", "B) 2", "C) 10", "D) 0"], "correct_answer": "B"},
            {"question": "What is the hypotenuse of a right-angled triangle with sides 3 and 4?", "options": ["A) 5", "B) 6", "C) 7", "D) 8"], "correct_answer": "A"}
        ],
        "physics": [
            {"question": "What is the acceleration due to gravity on Earth?", "options": ["A) 9.8 m/s^2", "B) 8.9 m/s^2", "C) 10 m/s^2", "D) 9.8 cm/s^2"], "correct_answer": "A"},
            {"question": "What is Newton's second law of motion?", "options": ["A) F = ma", "B) E = mc^2", "C) P = mv", "D) v = u + at"], "correct_answer": "A"},
            {"question": "What is the unit of electrical resistance?", "options": ["A) Ohm", "B) Ampere", "C) Volt", "D) Watt"], "correct_answer": "A"},
            {"question": "What form of energy is stored in a compressed spring?", "options": ["A) Kinetic", "B) Thermal", "C) Potential", "D) Chemical"], "correct_answer": "C"},
            {"question": "What is the speed of light in vacuum?", "options": ["A) 3 x 10^8 m/s", "B) 3 x 10^6 m/s", "C) 3 x 10^5 m/s", "D) 1.5 x 10^8 m/s"], "correct_answer": "A"}
        ],
        "chemistry": [
            {"question": "What is the chemical symbol for Gold?", "options": ["A) Gd", "B) Au", "C) Ag", "D) Fe"], "correct_answer": "B"},
            {"question": "What is the pH of pure water?", "options": ["A) 5", "B) 6", "C) 7", "D) 8"], "correct_answer": "C"},
            {"question": "Which gas is most abundant in Earth's atmosphere?", "options": ["A) Oxygen", "B) Nitrogen", "C) Carbon Dioxide", "D) Hydrogen"], "correct_answer": "B"},
            {"question": "What is the chemical formula for table salt?", "options": ["A) NaCl", "B) HCl", "C) NaOH", "D) H2O"], "correct_answer": "A"},
            {"question": "What is the atomic number of Hydrogen?", "options": ["A) 1", "B) 2", "C) 3", "D) 4"], "correct_answer": "A"}
        ]
    }
    
    topic_lower = topic.lower()
    selected_questions = []
    if "math" in topic_lower or "algebra" in topic_lower or "calculus" in topic_lower:
        selected_questions = topics["math"]
    elif "phys" in topic_lower or "mechanics" in topic_lower or "gravity" in topic_lower:
        selected_questions = topics["physics"]
    elif "chem" in topic_lower or "acid" in topic_lower or "element" in topic_lower:
        selected_questions = topics["chemistry"]
    else:
        selected_questions = [
            {"question": f"Which of the following is a key concept in {topic}?", "options": ["A) Theory A", "B) Concept B", "C) Principle C", "D) Method D"], "correct_answer": "B"},
            {"question": f"What is the primary goal of studying {topic}?", "options": ["A) Memorization", "B) Deep Understanding", "C) Passing tests only", "D) None of the above"], "correct_answer": "B"},
            {"question": f"In {topic}, which technique is most widely used?", "options": ["A) Analysis", "B) Synthesis", "C) Evaluation", "D) All of the above"], "correct_answer": "D"},
            {"question": f"What is a common misconception about {topic}?", "options": ["A) It is easy", "B) It has no real-world application", "C) It is only for scientists", "D) Both B and C"], "correct_answer": "D"},
            {"question": f"Who is considered a pioneer or major contributor to the study of {topic}?", "options": ["A) Leading Researcher X", "B) Classic Theorist Y", "C) Practitioner Z", "D) Diverse scholars throughout history"], "correct_answer": "D"}
        ]
        
    if len(selected_questions) < num_questions:
        selected_questions = selected_questions * (num_questions // len(selected_questions) + 1)
        
    random.shuffle(selected_questions)
    return selected_questions[:num_questions]

def generate_test_ai(source_text, subject_name, num_questions):
    prompt = f"""
You are an expert educator. Based on the study material/topic details provided below, generate a mock test with exactly {num_questions} multiple-choice questions.

Study Material/Topic:
{source_text}

You must respond ONLY with a JSON list containing exactly {num_questions} elements.
Each element must be a JSON object with the exact keys:
- "question": string, the multiple-choice question.
- "options": list of exactly 4 strings. The options must begin with "A) ", "B) ", "C) ", "D) ".
- "correct_answer": string, must be exactly "A", "B", "C", or "D".

Do not write any introductory text, markdown code blocks, or explanations. Respond with only the JSON string.
"""
    try:
        response = ollama.chat(
            model="llama3.2",
            messages=[{"role": "user", "content": prompt}]
        )
        content = response["message"]["content"].strip()
        
        if content.startswith("```"):
            lines = content.split("\n")
            if lines[0].startswith("```"):
                lines = lines[1:]
            if lines[-1].startswith("```"):
                lines = lines[:-1]
            content = "\n".join(lines).strip()
            
        questions = json.loads(content)
        if isinstance(questions, list) and len(questions) > 0:
            validated = []
            for q in questions:
                if "question" in q and "options" in q and "correct_answer" in q:
                    if len(q["options"]) == 4 and q["correct_answer"] in ["A", "B", "C", "D"]:
                        validated.append(q)
            if len(validated) >= num_questions:
                return validated[:num_questions]
            elif len(validated) > 0:
                return validated
    except Exception as e:
        print(f"Ollama generation failed or returned invalid JSON: {e}")
        
    return generate_fallback_questions(subject_name, num_questions)

def save_test_attempt(subject, score, total):
    data = load_data()
    attempts = data.get("test_attempts", [])
    
    new_id = 1 if not attempts else max(int(a.get("id", 0)) for a in attempts) + 1
    date_str = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")
    
    attempt = {
        "id": new_id,
        "date": date_str,
        "subject": subject,
        "score": score,
        "total": total,
        "percentage": (score / total) * 100 if total > 0 else 0
    }
    attempts.append(attempt)
    data["test_attempts"] = attempts
    save_data(data)

def get_attempts_df():
    data = load_data()
    attempts = data.get("test_attempts", [])
    rows = []
    for a in attempts:
        rows.append([
            a.get("id"),
            a.get("date"),
            a.get("subject"),
            a.get("score"),
            a.get("total"),
            f"{a.get('percentage', 0.0):.1f}%"
        ])
    return rows

def clear_attempts():
    data = load_data()
    data["test_attempts"] = []
    save_data(data)
    return get_attempts_df(), get_dashboard_metrics_html(), generate_dashboard_charts(), "Test history cleared."

# -------------------------
# Quiz State Transitions
# -------------------------
def start_test_flow(source_type, text_subject, pdf_file, num_questions):
    if source_type == "Uploaded PDF Content":
        if pdf_file is None:
            return (
                [], 0, 0, [], "", 
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
                "", "", gr.update(choices=[]), "⚠️ Please upload a PDF file first."
            )
        source_text = read_pdf(pdf_file)
        if not source_text.strip():
            return (
                [], 0, 0, [], "", 
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
                "", "", gr.update(choices=[]), "⚠️ No readable text found in the PDF."
            )
        subject_name = f"PDF: {os.path.basename(pdf_file.name)}"
    else:
        if not text_subject.strip():
            return (
                [], 0, 0, [], "", 
                gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
                "", "", gr.update(choices=[]), "⚠️ Please enter a subject or topic."
            )
        source_text = text_subject
        subject_name = text_subject
        
    questions = generate_test_ai(source_text, subject_name, int(num_questions))
    
    if not questions:
        return (
            [], 0, 0, [], "", 
            gr.update(visible=True), gr.update(visible=False), gr.update(visible=False),
            "", "", gr.update(choices=[]), "⚠️ Failed to generate test questions. Try again."
        )
        
    first_q = questions[0]
    q_num_text = f"### Question 1 of {len(questions)}"
    q_text = first_q["question"]
    choices = first_q["options"]
    
    return (
        questions, 0, 0, [], subject_name,
        gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
        q_num_text, q_text, gr.update(choices=choices, value=None), ""
    )

def submit_answer_flow(selected_option, questions, index, score, answers, subject):
    if not selected_option:
        return (
            index, score, answers,
            gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
            f"### Question {index + 1} of {len(questions)}",
            questions[index]["question"],
            gr.update(choices=questions[index]["options"], value=None),
            "⚠️ Please select an option before proceeding.",
            ""
        )
        
    chosen_letter = selected_option[0]
    correct_letter = questions[index]["correct_answer"]
    
    is_correct = (chosen_letter == correct_letter)
    new_score = score + 1 if is_correct else score
    new_answers = answers + [selected_option]
    
    next_index = index + 1
    
    if next_index < len(questions):
        next_q = questions[next_index]
        q_num_text = f"### Question {next_index + 1} of {len(questions)}"
        q_text = next_q["question"]
        choices = next_q["options"]
        
        return (
            next_index, new_score, new_answers,
            gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
            q_num_text, q_text, gr.update(choices=choices, value=None),
            "", ""
        )
    else:
        save_test_attempt(subject, new_score, len(questions))
        percentage = (new_score / len(questions)) * 100
        
        feedback = f"""## 🎓 Test Results
### Score: **{new_score} / {len(questions)}** ({percentage:.1f}%)

---
### 📝 Detailed Review
"""
        for i, q in enumerate(questions):
            user_ans = new_answers[i]
            correct_ans_letter = q["correct_answer"]
            
            correct_ans_text = ""
            for opt in q["options"]:
                if opt.startswith(correct_ans_letter):
                    correct_ans_text = opt
                    break
                    
            status_icon = "✅" if user_ans[0] == correct_ans_letter else "❌"
            feedback += f"""
**Question {i+1}:** {q['question']}
- {status_icon} **Your Answer:** {user_ans}
- 💡 **Correct Answer:** {correct_ans_text}

"""
        
        return (
            next_index, new_score, new_answers,
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
            "", "", gr.update(choices=[], value=None),
            "", feedback
        )

def return_to_setup():
    return (
        gr.update(visible=True),
        gr.update(visible=False),
        gr.update(visible=False)
    )

# -------------------------
# Dashboard Data & Charts
# -------------------------
def get_dashboard_metrics_html():
    data = load_data()
    attempts = data.get("test_attempts", [])
    goals = data.get("goals", [])
    
    total_tests = len(attempts)
    avg_score = sum(a.get("percentage", 0.0) for a in attempts) / total_tests if total_tests > 0 else 0
    highest_score = max(a.get("percentage", 0.0) for a in attempts) if total_tests > 0 else 0
    
    total_goals = len(goals)
    completed_goals = sum(1 for g in goals if g.get("completed", False))
    completion_rate = (completed_goals / total_goals) * 100 if total_goals > 0 else 0
    total_est = sum(float(g.get("estimated_hours", 0.0)) for g in goals)
    total_act = sum(float(g.get("actual_hours", 0.0)) for g in goals)
    
    html = f"""
    <div style="display: flex; gap: 15px; flex-wrap: wrap; margin-bottom: 20px; justify-content: space-between;">
        <div style="flex: 1; min-width: 180px; background: linear-gradient(135deg, #1e293b, #334155); border: 1px solid #475569; border-radius: 12px; padding: 15px; text-align: center; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="color: #94a3b8; font-size: 0.85rem; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; font-family: sans-serif;">Mock Tests Taken</div>
            <div style="color: #2dd4bf; font-size: 2rem; font-weight: 800; font-family: sans-serif;">{total_tests}</div>
        </div>
        <div style="flex: 1; min-width: 180px; background: linear-gradient(135deg, #1e293b, #334155); border: 1px solid #475569; border-radius: 12px; padding: 15px; text-align: center; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="color: #94a3b8; font-size: 0.85rem; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; font-family: sans-serif;">Average Score</div>
            <div style="color: #818cf8; font-size: 2rem; font-weight: 800; font-family: sans-serif;">{avg_score:.1f}%</div>
        </div>
        <div style="flex: 1; min-width: 180px; background: linear-gradient(135deg, #1e293b, #334155); border: 1px solid #475569; border-radius: 12px; padding: 15px; text-align: center; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="color: #94a3b8; font-size: 0.85rem; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; font-family: sans-serif;">Highest Score</div>
            <div style="color: #a855f7; font-size: 2rem; font-weight: 800; font-family: sans-serif;">{highest_score:.1f}%</div>
        </div>
        <div style="flex: 1; min-width: 180px; background: linear-gradient(135deg, #1e293b, #334155); border: 1px solid #475569; border-radius: 12px; padding: 15px; text-align: center; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="color: #94a3b8; font-size: 0.85rem; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; font-family: sans-serif;">Study Hours (Act/Est)</div>
            <div style="color: #34d399; font-size: 2rem; font-weight: 800; font-family: sans-serif;">{total_act:.1f}/{total_est:.1f}h</div>
        </div>
        <div style="flex: 1; min-width: 180px; background: linear-gradient(135deg, #1e293b, #334155); border: 1px solid #475569; border-radius: 12px; padding: 15px; text-align: center; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.1);">
            <div style="color: #94a3b8; font-size: 0.85rem; font-weight: 600; text-transform: uppercase; margin-bottom: 5px; font-family: sans-serif;">Goal Completion</div>
            <div style="color: #fbbf24; font-size: 2rem; font-weight: 800; font-family: sans-serif;">{completion_rate:.1f}%</div>
        </div>
    </div>
    """
    return html

def generate_dashboard_charts():
    data = load_data()
    attempts = data.get("test_attempts", [])
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)
    fig.patch.set_facecolor('#0f172a')
    
    text_color = '#f8fafc'
    
    ax1 = axes[0]
    ax1.set_facecolor('#1e293b')
    if attempts:
        indices = list(range(1, len(attempts) + 1))
        percentages = [a.get("percentage", 0.0) for a in attempts]
        
        ax1.plot(indices, percentages, marker='o', color='#2dd4bf', linewidth=2.5, markersize=8)
        ax1.fill_between(indices, percentages, color='#2dd4bf', alpha=0.15)
        
        ax1.set_title('Test Score Trends', fontsize=12, fontweight='bold', color=text_color, pad=12)
        ax1.set_xlabel('Attempt Number', fontsize=10, color='#cbd5e1')
        ax1.set_ylabel('Percentage (%)', fontsize=10, color='#cbd5e1')
        ax1.set_ylim(-5, 105)
        ax1.set_xticks(indices)
        ax1.tick_params(colors='#cbd5e1')
        ax1.grid(True, linestyle='--', alpha=0.2, color='#475569')
    else:
        ax1.text(0.5, 0.5, "No test attempts yet.\nTake a test in the 'Mock Test' tab!", 
                 ha='center', va='center', fontsize=11, color='#cbd5e1', transform=ax1.transAxes, style='italic')
        ax1.set_title('Test Score Trends', fontsize=12, fontweight='bold', color=text_color, pad=12)
        ax1.set_xticks([])
        ax1.set_yticks([])
        ax1.tick_params(colors='#cbd5e1')
        
    ax2 = axes[1]
    ax2.set_facecolor('#1e293b')
    if attempts:
        sub_map = {}
        for a in attempts:
            sub = a.get("subject", "General")
            if len(sub) > 15:
                sub = sub[:12] + "..."
            pct = a.get("percentage", 0.0)
            if sub not in sub_map:
                sub_map[sub] = []
            sub_map[sub].append(pct)
            
        avg_subs = {s: sum(p)/len(p) for s, p in sub_map.items()}
        sorted_subs = sorted(avg_subs.items(), key=lambda x: x[1])
        
        subjects = [s[0] for s in sorted_subs]
        avg_scores = [s[1] for s in sorted_subs]
        
        bars = ax2.barh(subjects, avg_scores, color='#818cf8', height=0.5, alpha=0.9)
        
        for bar in bars:
            width = bar.get_width()
            ax2.annotate(f'{width:.1f}%',
                        xy=(width, bar.get_y() + bar.get_height() / 2),
                        xytext=(3, 0),
                        textcoords="offset points",
                        ha='left', va='center', fontsize=9, fontweight='semibold', color='#f8fafc')
                        
        ax2.set_title('Average Score by Subject', fontsize=12, fontweight='bold', color=text_color, pad=12)
        ax2.set_xlabel('Average Score (%)', fontsize=10, color='#cbd5e1')
        ax2.set_xlim(0, 115)
        ax2.tick_params(colors='#cbd5e1')
        ax2.grid(True, axis='x', linestyle='--', alpha=0.2, color='#475569')
    else:
        ax2.text(0.5, 0.5, "No subject data yet.", 
                 ha='center', va='center', fontsize=11, color='#cbd5e1', transform=ax2.transAxes, style='italic')
        ax2.set_title('Average Score by Subject', fontsize=12, fontweight='bold', color=text_color, pad=12)
        ax2.set_xticks([])
        ax2.set_yticks([])
        ax2.tick_params(colors='#cbd5e1')
        
    for ax in [ax1, ax2]:
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_color('#334155')
        ax.spines['bottom'].set_color('#334155')
        
    plt.tight_layout()
    chart_path = "performance_dashboard.png"
    plt.savefig(chart_path, facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close()
    return chart_path

def load_dashboard_data():
    metrics = get_dashboard_metrics_html()
    chart = generate_dashboard_charts()
    df = get_attempts_df()
    return metrics, chart, df

def init_all_data():
    df_rows, summary, progress_chart = init_data()
    metrics, dashboard_chart, attempts_df = load_dashboard_data()
    return df_rows, summary, progress_chart, metrics, dashboard_chart, attempts_df

# -------------------------
# Custom styling CSS
# -------------------------
custom_css = """
.gradio-container {
    background-color: #0f172a !important;
    color: #e2e8f0 !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
}
/* Force text colors for all markdown prose, lists, headings, and strong text */
.prose, .prose p, .prose li, .prose h1, .prose h2, .prose h3, .prose h4, .prose strong, .prose span, .markdown-text, .markdown-text * {
    color: #f1f5f9 !important;
}
/* Force text colors for HTML headings */
h1, h2, h3, h4, h5, h6 {
    color: #f8fafc !important;
}
/* Ensure form labels and info text are highly visible */
.form-label, label, label span {
    color: #cbd5e1 !important;
}
.app-header {
    text-align: center;
    background: linear-gradient(135deg, #a855f7, #6366f1);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
    font-size: 2.8rem !important;
    font-weight: 800 !important;
    margin-bottom: 2px !important;
}
.app-subheader {
    text-align: center;
    color: #94a3b8 !important;
    font-size: 1.1rem !important;
    margin-bottom: 30px !important;
}
.card-panel {
    background-color: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 16px !important;
    padding: 24px !important;
    box-shadow: 0 4px 20px -2px rgba(0, 0, 0, 0.25) !important;
}
.action-btn {
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 700 !important;
    font-size: 1rem !important;
    padding: 10px 20px !important;
    transition: all 0.2s ease-in-out !important;
    box-shadow: 0 4px 10px rgba(99, 102, 241, 0.3) !important;
}
.action-btn:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 10px 20px rgba(99, 102, 241, 0.45) !important;
}
.sec-btn {
    background-color: #334155 !important;
    color: #f8fafc !important;
    border: 1px solid #475569 !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease-in-out !important;
}
.sec-btn:hover {
    background-color: #475569 !important;
}
.danger-btn {
    background-color: #ef4444 !important;
    color: white !important;
    border: none !important;
    border-radius: 10px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease-in-out !important;
}
.danger-btn:hover {
    background-color: #dc2626 !important;
}
.tab-nav button {
    font-weight: 600 !important;
    font-size: 1.1rem !important;
    color: #94a3b8 !important;
}
.tab-nav button.selected {
    color: #a855f7 !important;
    border-bottom-color: #a855f7 !important;
}
"""

# -------------------------
# Gradio UI Layout
# -------------------------
with gr.Blocks(title="Smart Student Study Planner") as demo:

    gr.HTML("<h1 class='app-header'>📚 Smart Student Study Planner</h1>")
    gr.HTML("<p class='app-subheader'>Organize your studies, query materials using AI, and track weekly goals</p>")

    with gr.Tab("AI Study Assistant"):
        with gr.Row(elem_classes=["card-panel"]):
            with gr.Column(scale=1):
                name = gr.Textbox(label="Student Name", placeholder="Enter your name...")
                pdf = gr.File(label="Upload PDF Material", file_types=[".pdf"])
                question = gr.Textbox(label="Ask Question", placeholder="What do you want to learn from the material?")
                ask = gr.Button("Ask AI Assistant ⚡", elem_classes=["action-btn"])
            with gr.Column(scale=1):
                answer = gr.Textbox(label="AI Answer", lines=16, interactive=False, placeholder="AI's answer will appear here...")
                
        ask.click(
            process,
            inputs=[name, pdf, question],
            outputs=answer
        )

    with gr.Tab("Daily Planner"):
        with gr.Row(elem_classes=["card-panel"]):
            with gr.Column(scale=1):
                student = gr.Textbox(label="Student Name", placeholder="Enter student name...")
                subject = gr.Textbox(label="Subject", placeholder="Subject to study...")
                hours = gr.Slider(1, 12, value=2, step=1, label="Target Study Hours")
                
                with gr.Row():
                    save = gr.Button("Save Plan", elem_classes=["action-btn"])
                    view = gr.Button("View Saved Plan", elem_classes=["sec-btn"])
                    
            with gr.Column(scale=1):
                output = gr.Textbox(label="Daily Study Plan Output", lines=13, interactive=False)

        save.click(
            save_plan,
            inputs=[student, subject, hours],
            outputs=output
        )

        view.click(
            view_plan,
            outputs=output
        )

    with gr.Tab("Weekly Goals & Progress"):
        # Graph & Summary Row
        with gr.Row(elem_classes=["card-panel"]):
            with gr.Column(scale=3):
                progress_chart = gr.Image(label="Weekly Study Progress", interactive=False)
            with gr.Column(scale=2):
                summary_md = gr.Markdown("### 📊 Loading Weekly Summary...")

        # Interactive Goal Grid Row
        with gr.Row(elem_classes=["card-panel"]):
            with gr.Column(scale=1):
                gr.HTML("<h3>✍️ Add New Goal</h3>")
                new_goal_desc = gr.Textbox(label="Goal Description", placeholder="e.g. Read Physics Chapter 3")
                new_goal_day = gr.Dropdown(
                    choices=["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"], 
                    value="Monday", 
                    label="Target Day"
                )
                new_goal_hours = gr.Slider(0.5, 12, step=0.5, value=2, label="Estimated Study Hours")
                add_btn = gr.Button("Add Goal", elem_classes=["action-btn"])
                
            with gr.Column(scale=2):
                gr.HTML("<h3>📋 Weekly Goals Grid</h3>")
                gr.HTML("<p style='color: #94a3b8; font-size: 0.9rem;'>You can edit 'Act. Hours' and 'Completed' directly in the table below, then click 'Save Table Changes'.</p>")
                
                goals_df = gr.Dataframe(
                    headers=["ID", "Goal", "Day", "Est. Hours", "Act. Hours", "Completed"],
                    datatype=["number", "str", "str", "number", "number", "bool"],
                    interactive=True
                )
                
                with gr.Row():
                    save_btn = gr.Button("Save Table Changes", elem_classes=["action-btn"])
                    clear_btn = gr.Button("Clear All Goals", elem_classes=["danger-btn"])
                    
                status_txt = gr.Textbox(label="Status Console", interactive=False, placeholder="Status messages...")

        # Actions for Weekly tab
        add_btn.click(
            add_goal,
            inputs=[new_goal_desc, new_goal_day, new_goal_hours],
            outputs=[goals_df, status_txt, progress_chart]
        ).then(
            get_progress_summary,
            outputs=summary_md
        )

        save_btn.click(
            save_goals_from_df,
            inputs=[goals_df],
            outputs=[status_txt, progress_chart]
        ).then(
            get_progress_summary,
            outputs=summary_md
        )

        clear_btn.click(
            clear_all_goals,
            outputs=[goals_df, status_txt, progress_chart]
        ).then(
            get_progress_summary,
            outputs=summary_md
        )

    # Define state components inside gr.Blocks
    test_questions = gr.State([])
    current_q_idx = gr.State(0)
    test_score = gr.State(0)
    user_answers = gr.State([])
    test_subject = gr.State("")

    with gr.Tab("Mock Test"):
        # Setup Container
        with gr.Column(visible=True) as setup_container:
            with gr.Row(elem_classes=["card-panel"]):
                with gr.Column():
                    gr.HTML("<h3>📝 Generate Mock Test</h3>")
                    test_source = gr.Dropdown(
                        choices=["Custom Text Subject/Topic", "Uploaded PDF Content"],
                        value="Custom Text Subject/Topic",
                        label="Test Material Source"
                    )
                    text_subject_input = gr.Textbox(
                        label="Subject / Topic",
                        placeholder="e.g., Physics - Thermodynamics, Organic Chemistry, Calculus"
                    )
                    num_questions_input = gr.Slider(
                        minimum=3, maximum=10, step=1, value=5,
                        label="Number of Questions"
                    )
                    generate_btn = gr.Button("Generate Test 🚀", elem_classes=["action-btn"])
                    setup_status = gr.Textbox(label="Status Log", interactive=False, placeholder="Waiting...")
                    
        # Quiz Play Container
        with gr.Column(visible=False) as quiz_container:
            with gr.Row(elem_classes=["card-panel"]):
                with gr.Column():
                    q_num_display = gr.Markdown("### Question 1 of 5")
                    q_text_display = gr.Markdown("What is the speed of light?")
                    
                    q_options_radio = gr.Radio(
                        choices=[],
                        label="Choose the correct option:"
                    )
                    
                    submit_answer_btn = gr.Button("Submit Answer & Next ⚡", elem_classes=["action-btn"])
                    quiz_status = gr.Textbox(label="Quiz Console", interactive=False, placeholder="")
                    
        # Results Container
        with gr.Column(visible=False) as results_container:
            with gr.Row(elem_classes=["card-panel"]):
                with gr.Column():
                    results_md_display = gr.Markdown("## Test Completed!")
                    restart_test_btn = gr.Button("Start New Test 🔁", elem_classes=["sec-btn"])

        # Quiz Navigation Clicks
        generate_btn.click(
            start_test_flow,
            inputs=[test_source, text_subject_input, pdf, num_questions_input],
            outputs=[
                test_questions, current_q_idx, test_score, user_answers, test_subject,
                setup_container, quiz_container, results_container,
                q_num_display, q_text_display, q_options_radio, setup_status
            ]
        )
        
        submit_answer_btn.click(
            submit_answer_flow,
            inputs=[q_options_radio, test_questions, current_q_idx, test_score, user_answers, test_subject],
            outputs=[
                current_q_idx, test_score, user_answers,
                setup_container, quiz_container, results_container,
                q_num_display, q_text_display, q_options_radio, quiz_status, results_md_display
            ]
        )
        
        restart_test_btn.click(
            return_to_setup,
            outputs=[setup_container, quiz_container, results_container]
        )

    with gr.Tab("Performance Dashboard") as dashboard_tab:
        with gr.Row(elem_classes=["card-panel"]):
            metrics_html_display = gr.HTML("Loading metrics dashboard...")
            
        with gr.Row(elem_classes=["card-panel"]):
            with gr.Column(scale=3):
                dashboard_charts_display = gr.Image(label="Performance Visualizations", interactive=False)
            with gr.Column(scale=2):
                gr.HTML("<h3>📊 Mock Test History</h3>")
                attempts_history_df = gr.Dataframe(
                    headers=["ID", "Date", "Subject/Topic", "Score", "Total", "Percentage"],
                    datatype=["number", "str", "str", "number", "number", "str"],
                    interactive=False
                )
                with gr.Row():
                    refresh_dash_btn = gr.Button("Refresh Dashboard 🔄", elem_classes=["sec-btn"])
                    reset_dash_btn = gr.Button("Reset Test History ⚠️", elem_classes=["danger-btn"])
                dash_status_txt = gr.Textbox(label="Dashboard Status", interactive=False)

        # Connect Refresh & Reset
        refresh_dash_btn.click(
            load_dashboard_data,
            outputs=[metrics_html_display, dashboard_charts_display, attempts_history_df]
        )
        
        reset_dash_btn.click(
            clear_attempts,
            outputs=[attempts_history_df, metrics_html_display, dashboard_charts_display, dash_status_txt]
        )
        
        # When tab is selected, reload dashboard data
        dashboard_tab.select(
            load_dashboard_data,
            outputs=[metrics_html_display, dashboard_charts_display, attempts_history_df]
        )

    # Initial data load
    demo.load(
        init_all_data,
        outputs=[
            goals_df, summary_md, progress_chart,
            metrics_html_display, dashboard_charts_display, attempts_history_df
        ]
    )

if __name__ == "__main__":
    demo.launch(css=custom_css)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


findfont: Failed to find font weight semibold, now using 700.
